In [7]:
#library(ggplot2)
#library(grid)
#library(gridExtra)
#library(reshape2)
library(data.table)
#library(rsnps)
#library(rtracklayer)
#library(biomaRt)
library(dplyr)
#library(bigsnpr)
library(vcfR)
library(pegas)

Loading required package: ape


Attaching package: ‘pegas’


The following object is masked from ‘package:ape’:

    mst


The following objects are masked from ‘package:vcfR’:

    getINFO, write.vcf




In [6]:
#install.packages("pegas")

Installing package into ‘/u/home/a/afcarrol/R/x86_64-pc-linux-gnu-library-RH7/4.1.0’
(as ‘lib’ is unspecified)



In [25]:
pgs <- read.table("/u/project/pasaniuc/kangchen/2022-ace-analysis/out/02-pgs/processed-weights/ASD2019.tsv")


In [26]:
colnames(pgs) <- pgs[1,]
pgs <- pgs[-1,]

In [27]:
head(pgs[,])

,CHROM,POS,ALT,REF,WEIGHT
,<chr>,<chr>,<chr>,<chr>,<chr>
2,1,979472,C,G,0.0349037
3,1,1161955,T,G,0.0638007
4,1,1967499,A,G,-0.0295009
5,1,1985090,T,C,-0.0355961
6,1,2010975,T,C,-0.0349953
7,1,2024545,T,C,0.036303


In [28]:
colnames(pgs) <- c("CHROM", "POS", "A1", "A2", "WEIGHT")

pgs$id <- paste0("chr", pgs$CHR, ":", pgs$BP, ":", pgs$A2, ":", pgs$A1 )

#pgs$BETA <- as.numeric(pgs$BETA)

In [29]:
head(pgs)

,CHROM,POS,A1,A2,WEIGHT,id
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
2,1,979472,C,G,0.0349037,chr1::G:C
3,1,1161955,T,G,0.0638007,chr1::G:T
4,1,1967499,A,G,-0.0295009,chr1::G:A
5,1,1985090,T,C,-0.0355961,chr1::C:T
6,1,2010975,T,C,-0.0349953,chr1::C:T
7,1,2024545,T,C,0.036303,chr1::C:T


In [43]:
list.files(path="/u/project/geschwind/shared/GenomicDatasets-processed/ACE-ANALYSIS/freeze0/SPARK/imputed/")

[1] "chr1.log"   "chr1.pgen"  "chr1.psam"  "chr1.pvar"  "chr10.log" 
 [6] "chr10.pgen" "chr10.psam" "chr10.pvar" "chr11.log"  "chr11.pgen"
[11] "chr11.psam" "chr11.pvar" "chr12.log"  "chr12.pgen" "chr12.psam"
[16] "chr12.pvar" "chr13.log"  "chr13.pgen" "chr13.psam" "chr13.pvar"
[21] "chr14.log"  "chr14.pgen" "chr14.psam" "chr14.pvar" "chr15.log" 
[26] "chr15.pgen" "chr15.psam" "chr15.pvar" "chr16.log"  "chr16.pgen"
[31] "chr16.psam" "chr16.pvar" "chr17.log"  "chr17.pgen" "chr17.psam"
[36] "chr17.pvar" "chr18.log"  "chr18.pgen" "chr18.psam" "chr18.pvar"
[41] "chr19.log"  "chr19.pgen" "chr19.psam" "chr19.pvar" "chr2.log"  
[46] "chr2.pgen"  "chr2.psam"  "chr2.pvar"  "chr20.log"  "chr20.pgen"
[51] "chr20.psam" "chr20.pvar" "chr21.log"  "chr21.pgen" "chr21.psam"
[56] "chr21.pvar" "chr22.log"  "chr22.pgen" "chr22.psam" "chr22.pvar"
[61] "chr3.log"   "chr3.pgen"  "chr3.psam"  "chr3.pvar"  "chr4.log"  
[66] "chr4.pgen"  "chr4.psam"  "chr4.pvar"  "chr5.log"   "chr5.pgen" 
[71] "chr5.psam"  "chr5.pvar"  "chr6.log"   "chr6.pgen"  "chr6.psam" 
[76] "chr6.pvar"  "chr7.log"   "chr7.pgen"  "chr7.psam"  "chr7.pvar" 
[81] "chr8.log"   "chr8.pgen"  "chr8.psam"  "chr8.pvar"  "chr9.log"  
[86] "chr9.pgen"  "chr9.psam"  "chr9.pvar"

In [1]:


t1 <- read_pgen("/u/project/geschwind/shared/GenomicDatasets-processed/ACE-ANALYSIS/freeze0/SPARK/imputed/chr1", variantidx = NULL, meanimpute = F)


ERROR: Error in read_pgen("/u/project/geschwind/shared/GenomicDatasets-processed/ACE-ANALYSIS/freeze0/SPARK/imputed/chr1", : could not find function "read_pgen"


In [ ]:
total_sum <- list()
vars <- c()

for(i in 1:22){
  
vcf <- as.matrix(vcfR2genlight(read.vcfR(paste0("mexico_vcfs_trimmed_hg38/chr",i, ".vcf"))))

vcf <- vcf[,colnames(vcf) %in% pgs$id]
vars <- c(vars, ncol(vcf))
for(col_head in colnames(vcf)){
  vcf[,col_head]<- vcf[,col_head]*pgs[pgs$id ==col_head, "BETA"]
}
total_sum[[i]] <- rowSums(vcf)

}

In [ ]:
total_vars <- c()

for(i in 1:22){
 print(i) 
  
vcf <- as.matrix(vcfR2genlight(read.vcfR(paste0("mexico_vcfs_trimmed_hg38/chr",i, ".vcf"))))

total_vars <- c(total_vars, ncol(vcf))

}

In [ ]:
sum_df <- do.call(rbind,total_sum)

total_score <- as.data.frame(colSums(sum_df))

colnames(total_score) <- "pgs_score"

total_score$IID <- rownames(total_score)

write.csv(total_score, "manual_pgs_2.csv",col.names = T,row.names = F)